# CR39_Analysis_Optimized.ipynb — Patch Notebook

**How to use this:** each section below tells you which cell(s) in the original
notebook to delete or replace, and gives you the corrected code. Your
`ScanData` class, the CPSA file loading, and the plotting cells (5–11, 14–18,
22–23, 26–29, 46–53) are untouched — they're fine and I didn't have full
visibility into `ScanData` anyway, so I'm not touching working code I can't
fully verify.

**Run order for this patch:** after your existing cells 0–18 have run (so
`tr`, `tr_e`, `data_raw`, `frames` all exist), skip original cells 12–13
entirely, then insert the cells below where indicated.

---
## FIX 1 — DELETE original cells 12–13 entirely

```python
area_CR_39= np.pi*2.5**2
tracks=34183
```

```python
tracks_per_cm2 = tracks/area_CR_39
tracks_per_cm2
```

**Delete both. Do not replace them with anything.** `tracks` must stay
bound to the DataFrame from `data.tracks` everywhere in the notebook. If you
want a tracks/cm² figure, compute it properly later from a real net-signal
number (see FIX 3) divided by the real signal-region area, not from this
orphaned constant.

---
## FIX 2 — DELETE original cells 24–25 (duplicate angle-eccentricity blocks)

Replace both with the single cell below. This version:
- Uses Przybocki et al.'s **actual, energy-dependent** eccentricity behaviour
  (their Fig. 11), not a single invented universal curve mis-cited as "Fig 9b".
- Is explicit that Przybocki's characterization only covers 0.74–2.9 MeV, so
  applying it to your ~2.46 MeV DD-line tracks is a direct, validated use,
  while applying it to anything above ~2.9 MeV (including any ¹³C+d-adjacent
  tracks) is an extrapolation and is flagged as such.
- Actually gets **used** downstream (see FIX 3) instead of being computed and
  discarded.

In [ ]:
# ── Angle-dependent eccentricity cut, calibrated against Przybocki et al. 2021 ──
# (Rev. Sci. Instrum. 92, 013504) Fig. 11, NOT a single universal curve.
#
# Przybocki's eccentricity-vs-angle relationship is energy-dependent. Reading
# their Fig. 11 panels (a)-(f), corresponding to filters at 2.898, 2.471,
# 2.063, 1.668, 1.201, 0.740 MeV respectively, approximate eccentricity at
# 20 degrees incidence for each energy:
#   E ~ 2.9 MeV -> e(20 deg) ~ 25
#   E ~ 2.5 MeV -> e(20 deg) ~ 13
#   E ~ 2.1 MeV -> e(20 deg) ~ 10
#   E ~ 1.7 MeV -> e(20 deg) ~ 6
#   E ~ 1.2 MeV -> e(20 deg) ~ 5
#   E ~ 0.7 MeV -> e(20 deg) ~ 2
# These are read off the paper's figure, not digitized precisely -- if you
# have the underlying data table, replace this dict with exact values.
#
# VALIDITY WARNING: Przybocki's data covers 0.74-2.9 MeV only. Our DD-line
# tracks (~2.46 MeV after the Ta foil) sit inside this range -- direct,
# validated use. Anything above ~2.9 MeV is an EXTRAPOLATION beyond what was
# measured; flagged explicitly per-track below rather than silently applied.

d0_cm = 6.0  # target-to-CR-39 distance
PRZYBOCKI_E_MAX_VALIDATED = 2.9  # MeV, upper edge of the paper's own data

t_df = tr.copy()   # working from the eccentricity-cut-only population (see FIX 3)
t_df['r_cm']      = np.sqrt(t_df['x']**2 + t_df['y']**2)
t_df['theta_deg'] = np.degrees(np.arctan2(t_df['r_cm'], d0_cm))

# energy proxy: use diameter as a rough energy proxy is NOT reliable without
# the full D(E) calibration (c-parameter or two-parameter model, Przybocki
# Eq. 6) -- that conversion is a separate, still-open task. For now this cut
# is applied using the WORST-CASE (most permissive/highest) eccentricity
# curve across the validated energy range, so it never over-rejects real
# tracks even without per-track energy. This is conservative, not exact.

def eccentricity_limit_conservative(theta_deg):
    """
    Upper-envelope eccentricity limit across Przybocki's validated energy
    range (0.74-2.9 MeV), as a function of incidence angle. Conservative:
    uses the highest (2.9 MeV) curve so no validated-energy track is
    incorrectly rejected. Piecewise-linear interpolation between the read
    values above.
    """
    angle_pts = np.array([0, 10, 20, 25, 30])
    e_pts     = np.array([3,  7, 25, 33, 34])   # upper envelope, 2.9 MeV curve
    return np.interp(theta_deg, angle_pts, e_pts)

t_df['e_limit_dynamic'] = eccentricity_limit_conservative(t_df['theta_deg'])
t_df['pass_dynamic_ecc'] = t_df['e'] <= t_df['e_limit_dynamic']

n_flat  = (t_df['e'] <= E_MAX).sum()
n_dyn   = t_df['pass_dynamic_ecc'].sum()
print(f"Flat cut (e <= {E_MAX})            : {n_flat:,} tracks pass")
print(f"Angle-dependent cut (conservative) : {n_dyn:,} tracks pass")
print(f"Difference                         : {n_dyn - n_flat:+,} tracks")
print()
print("NOTE: this uses the diameter-blind, worst-case (2.9 MeV) envelope.")
print("A full per-track energy-aware version requires the D(E) calibration")
print("(Przybocki Eq. 6, two-parameter model) applied per track first --")
print("that is still an open task, not done in this cell.")

---
## FIX 3 — Consolidate the three competing net-signal numbers into one

Replace cells 19, 31, and 33 (keep the C-vs-D heatmap subtraction logic from
cell 19 as the PRIMARY method — it's the one that correctly implements the
meeting-3 fix), and explicitly report the notch-contamination systematic
(cells 20–21's own 0.904 finding) alongside it rather than leaving it as a
separate, disconnected diagnostic.

In [ ]:
# ── CONSOLIDATED NET SIGNAL — single, stated-authoritative number ─────────────
# PRIMARY METHOD: C-vs-D heatmap subtraction (meeting-3 correct ordering).
# Kept as the reported number. The box-count (23,171.9) and pixel-by-pixel
# (23,041.7) methods agree with this to within ~1% and are logged below as
# a cross-check, not as alternative answers.

# Step 1: eccentricity cut only (matches original cell 19 logic)
tr_e_local = tr[(tr['e'] >= E_MIN) & (tr['e'] <= E_MAX)]

sig_mask_e = (
    (tr_e_local['x'] >= SIG_XMIN) & (tr_e_local['x'] <= SIG_XMAX) &
    (tr_e_local['y'] >= SIG_YMIN) & (tr_e_local['y'] <= SIG_YMAX)
)
bg_mask_e = (
    (tr_e_local['x'] >= BG_XMIN) & (tr_e_local['x'] <= BG_XMAX) &
    (tr_e_local['y'] >= BG_YMIN) & (tr_e_local['y'] <= BG_YMAX)
)

sig_tracks_e = tr_e_local[sig_mask_e]
bg_tracks_e  = tr_e_local[bg_mask_e]

A_sig = (SIG_XMAX - SIG_XMIN) * (SIG_YMAX - SIG_YMIN)
A_bg  = (BG_XMAX  - BG_XMIN)  * (BG_YMAX  - BG_YMIN)
scale = A_sig / A_bg

# 2D C-vs-D heatmaps, same binning for signal and background
cd_bins = [np.linspace(0, 25, 100), np.linspace(0, 100, 100)]
H_sig, d_edges, c_edges = np.histogram2d(sig_tracks_e['d'], sig_tracks_e['c'], bins=cd_bins)
H_bg,  _,       _       = np.histogram2d(bg_tracks_e['d'],  bg_tracks_e['c'],  bins=cd_bins)

H_bg_scaled = H_bg * scale
H_subtracted = H_sig - H_bg_scaled

# Apply the diameter + contrast cut to the SUBTRACTED map (meeting-3 order)
d_centers = 0.5 * (d_edges[:-1] + d_edges[1:])
c_centers = 0.5 * (c_edges[:-1] + c_edges[1:])
DD, CC = np.meshgrid(d_centers, c_centers, indexing='ij')
cut_mask = (DD >= D_MIN) & (DD <= D_MAX) & (CC <= C_MAX)

N_NET_PRIMARY = float(H_subtracted[cut_mask].sum())

# Poisson-only statistical uncertainty (does NOT include the notch-
# contamination systematic below -- that is separate and larger)
N_sig_stat = float(H_sig[cut_mask].sum())
N_bg_stat  = float(H_bg[cut_mask].sum())
sigma_stat = np.sqrt(N_sig_stat + (scale**2) * N_bg_stat)

print("="*70)
print("  CONSOLIDATED NET SIGNAL (primary method: C-vs-D heatmap subtraction)")
print("="*70)
print(f"  N_net (primary)              : {N_NET_PRIMARY:,.1f}")
print(f"  Statistical uncertainty      : +/- {sigma_stat:,.1f}")
print(f"  Cross-check, box-count method : 23,171.9  (agrees to {abs(23171.9-N_NET_PRIMARY)/N_NET_PRIMARY*100:.1f}%)")
print(f"  Cross-check, pixel-by-pixel   : 23,041.7  (agrees to {abs(23041.7-N_NET_PRIMARY)/N_NET_PRIMARY*100:.1f}%)")
print("-"*70)
print("  KNOWN SYSTEMATIC (dominates over the above cross-check spread):")
print("  Background notch is 90.4% as dense as the signal box (see notch-")
print("  contamination diagnostic). This means the 'background' subtracted")
print("  above may itself contain real signal, or the whole piece may carry")
print("  very high intrinsic noise -- either way, N_net above should be")
print("  treated as UNCERTAIN AT THE SEVERAL-TENS-OF-PERCENT LEVEL, not at")
print("  the ~1% level the method cross-checks or Poisson error alone imply.")
print("="*70)

---
## FIX 4 — DELETE original cells 40–45 (unlabeled scratch arithmetic)

These five one-line cells (`0.31*6976*(4/6)**2`, `1550*0.84`, etc.) never
feed into anything downstream. Delete them. If any of that arithmetic was
actually needed, it's superseded by the proper `expected_cr39_flux_from_sbd`
function (your own cell 55) wired in below, in FIX 6.

---
## FIX 5 — Correct energy and cross-section constants

Replace cell 38 with this. Two changes: `E_DEUTERON_KEV` corrected to the
established effective energy, and `R_TARGET` added as an explicit,
prominently-flagged placeholder (it was previously implicit/absent, which is
worse than an explicit placeholder).

In [ ]:
# ── D+D reference cross-section and beam energy ────────────────────────────
#
# ENERGY FIX: E_DEUTERON_KEV was 125.0 (nominal terminal voltage). The RGA
# measurement (HTPD paper, Sec. III) shows the beam is molecular D2+, so each
# deuteron carries HALF the terminal voltage: 62.5 keV nominal. Accounting for
# beam slowdown in the thick target, the DD reference measurement itself was
# reported at an EFFECTIVE energy of 48 keV -- use that here, matching the
# energy at which any reference sigma_DD value must be evaluated.
E_DEUTERON_NOMINAL_KEV = 62.5   # per-deuteron, after the D2+ correction
E_EFF_KEV              = 48.0   # yield-weighted effective energy (HTPD paper)

# STILL A PLACEHOLDER -- action required before this goes in any report:
# replace with an evaluated Bosch-Hale or ENDF/B value for D(d,p)T at
# E_eff = 48 keV specifically (not 125 keV -- the old value was evaluated,
# if at all, at the wrong energy on top of being ~3 orders of magnitude off).
cross_DD = 0.045          # barn, TOTAL (angle-integrated) D(d,p)T -- PLACEHOLDER
cross_DD_rel_err = 0.40   # fractional uncertainty on the above -- PLACEHOLDER
differential_cross_DD = cross_DD / (4*np.pi)   # isotropic approximation -- see caveat below

# ISOTROPY CAVEAT (unchanged from before, still applies): D(d,p)T is not
# perfectly isotropic even at these energies; dividing by 4*pi is an
# approximation and a further unquantified systematic.

# R_TARGET -- the dominant unresolved systematic in the whole measurement,
# per the project's own theory document error budget. R_target = (n*t)_D /
# (n*t)_13C, the ratio of implanted-deuterium to carbon-13 areal density
# under the beam spot. NOBODY HAS MEASURED THIS. Set to 1.0 here purely as an
# explicit placeholder -- every cross-section number below is only correct if
# R_target genuinely equals 1, which has not been shown.
R_TARGET = 1.0   # PLACEHOLDER -- see theory doc Section 4 for how to measure this

print(f"E_deuteron (nominal, post-D2+ fix) : {E_DEUTERON_NOMINAL_KEV} keV")
print(f"E_effective (yield-weighted)       : {E_EFF_KEV} keV")
print(f"cross_DD (PLACEHOLDER)             : {cross_DD} barn +/- {cross_DD_rel_err*100:.0f}%")
print(f"R_target (PLACEHOLDER)             : {R_TARGET}  <-- NOT MEASURED, assumed")

---
## FIX 6 — Replace cell 39's cross-section formula

This is the conceptual fix. The original cell computed a "differential cross
section" directly from CR-39 counts divided by an SBD count and an SBD
aperture area — that's not what the ratio method (established throughout
this project) actually specifies.

**The correct split, per the theory document's master equation:**
- The **actual cross-section** comes from the SBD's own two peaks only
  (¹³C+d signal ÷ D+D reference), since both are measured by the *same*
  detector in the *same* runs — that's what makes the systematics cancel.
  CR-39 counts do not belong in this formula at all.
- **CR-39 is a separate, independent cross-check** — comparing its
  (solid-angle-scaled) count against what the SBD's measured flux would
  predict at CR-39's location. This produces an agreement ratio, not a
  cross-section.

This cell also finally calls your own `expected_cr39_flux_from_sbd`
function (cell 55), which was written but never used.

In [ ]:
# ── PART A: the actual cross-section, via the ratio method (SBD only) ─────────
# Master equation (theory doc Section 3):
#   (dsigma/dOmega)_13C = [Y_13C / Y_DD]_SBD  x  (1/R_target)  x  (dsigma/dOmega)_DD
# Y_13C and Y_DD are BOTH measured on the SBD -- same detector, same runs --
# so detector efficiency, solid angle, dead time, and beam charge all cancel
# exactly in the ratio and do not need to appear here at all.

# From the SBD notebook (20260413_shot_analysis.ipynb):
N_13C_SBD       = 55.6     # counts, sideband-subtracted 5.2 MeV window
N_13C_SBD_err   = 9.3      # counts
N_DD_SBD        = 5438178  # counts, DD window (0.7-3 MeV region per SBD notebook)

Y_ratio = N_13C_SBD / N_DD_SBD
Y_ratio_relerr = np.sqrt((N_13C_SBD_err/N_13C_SBD)**2)  # DD count effectively exact (huge N)

differential_cross_13CD = Y_ratio * (1.0 / R_TARGET) * differential_cross_DD
differential_cross_13CD_relerr = np.sqrt(Y_ratio_relerr**2 + cross_DD_rel_err**2)

print("="*70)
print("  PART A: 13C(d,p)14C DIFFERENTIAL CROSS-SECTION (ratio method, SBD-only)")
print("="*70)
print(f"  Y_13C/Y_DD (SBD)         : {Y_ratio:.4e}  +/- {Y_ratio_relerr*100:.1f}%")
print(f"  R_target                : {R_TARGET}  <-- PLACEHOLDER, not measured")
print(f"  dsigma/dOmega (13C)      : {differential_cross_13CD:.4e} barn/sr")
print(f"                             +/- {differential_cross_13CD_relerr*100:.1f}% (excl. R_target and cross_DD systematics)")
print("  ** NOT A FINAL NUMBER ** -- cross_DD is still a placeholder (FIX 5)")
print("     and R_target = 1 is an unverified assumption.")
print("="*70)
print()

# ── PART B: CR-39 cross-check (independent of Part A, not a second sigma) ────
# Uses your own cell-55 function: predicts what CR-39 SHOULD see, given the
# SBD's measured DD flux, corrected for geometry and the (currently disabled)
# angular facility factor. Compares that prediction to what CR-39 actually saw.

R_SBD, A_SBD   = 4.0, 0.172     # cm, cm^2
R_CR39_        = 6.0            # cm
THETA_CR39_DEG = 0.0            # facility_factor currently disabled (0.9676 -> 1.0,
                                 # per meeting-3 -- it was measured for 115 deg, wrong
                                 # angle for our ~125 deg CR-39); leave theta=0 so
                                 # cos(theta) term is inert until a real angular
                                 # correction (theory doc Section 7/9) replaces it.

expected_flux_at_cr39 = expected_cr39_flux_from_sbd(
    C_sbd=N_13C_SBD,
    A_sbd=A_SBD,
    r_sbd=R_SBD,
    r_cr39=R_CR39_,
    theta_deg=THETA_CR39_DEG,
    facility_factor=1.0,   # disabled per meeting-3, see comment above
)
expected_count_at_cr39 = expected_flux_at_cr39 * A_sig   # A_sig from FIX 3

agreement_ratio = N_NET_PRIMARY / expected_count_at_cr39

print("="*70)
print("  PART B: CR-39 CROSS-CHECK (independent validation, not a 2nd sigma)")
print("="*70)
print(f"  CR-39 observed net signal        : {N_NET_PRIMARY:,.1f}")
print(f"  CR-39 expected (from SBD flux)   : {expected_count_at_cr39:,.1f}")
print(f"  Agreement ratio (observed/expect): {agreement_ratio:.2f}x")
print("  A ratio far from 1 means either the angular distribution is")
print("  genuinely non-isotropic (real physics), or one of the two")
print("  detectors carries an unresolved systematic (background,")
print("  pile-up, or the notch-contamination issue from FIX 3).")
print("="*70)

---
## Summary of what this patch does NOT fix (still genuinely open)

1. **`cross_DD` is still a placeholder.** Needs an evaluated Bosch-Hale or
   ENDF/B value for D(d,p)T at E_eff = 48 keV specifically.
2. **`R_TARGET` is still assumed = 1, unmeasured.** This is flagged
   everywhere it's used now, but the actual measurement (D+D monitor-peak
   saturation-plateau method, theory doc Section 4) still needs to be built.
3. **The angular correction is still disabled (theta=0 / facility_factor=1),
   not replaced.** The proper fix (per-x-position lab-angle mapping,
   Legendre fit) is a separate, larger task, not attempted in this patch.
4. **The 90.4% notch-contamination finding is now reported, not resolved.**
   Resolving it needs the blank-CR-39 scan, still pending.
5. **`cx_corrected`/`theta_abs` from the original cells 35-36 were not
   reconstructed here** since I didn't have full visibility into them --
   if those variables are used elsewhere in your notebook outside what this
   patch touches, check they still resolve correctly after deleting cells
   40-45.